In [0]:
import os
import mlflow
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler

# CONFIG
SILVER_TABLE = "iotmlhealthcatalog.silver.vitaldbstream"
GOLD_TABLE = "iotmlhealthcatalog.gold.vitaldbstream"
CHECKPOINT = "/Volumes/iotmlhealthcatalog/default/iotbatch/_checkpoints/gold_stream"

# ===============================
# CONFIG MLflow (UC compatible)
# ===============================
TMP_PATH = "/Volumes/iotmlhealthcatalog/default/iotbatch/tmp"
os.environ["MLFLOW_DFS_TMP"] = TMP_PATH

# =========================
# LOAD MODEL (SPARK MODEL)
# =========================
model = mlflow.spark.load_model(
    "models:/iotmlhealthcatalog.gold.rf_vitaldb_for_streaming@production"
)

# =========================
# STREAM
# =========================
df = spark.readStream.table(SILVER_TABLE)

# =========================
# FEATURES
# =========================
df = df \
    .withColumn("shock_index",
        F.when(F.col("sbp") > 0, F.col("hr") / F.col("sbp")).otherwise(0)
    ) \
    .withColumn("hr_spo2_ratio",
        F.when(F.col("spo2") > 0, F.col("hr") / F.col("spo2")).otherwise(0)
    ) \
    .withColumn("is_low_sbp",
        F.when(F.col("sbp") < 90, 1.0).otherwise(0.0)
    ) \
    .withColumn("is_high_hr",
        F.when(F.col("hr") > 100, 1.0).otherwise(0.0)
    ) \
    .withColumn("is_low_spo2",
        F.when(F.col("spo2") < 92, 1.0).otherwise(0.0)
    )

features_cols = [
    "sbp","hr","spo2","temp",
    "shock_index","hr_spo2_ratio",
    "is_low_sbp","is_high_hr","is_low_spo2"
]

# =========================
# ASSEMBLER (IMPORTANT)
# =========================
#assembler = VectorAssembler(
#    inputCols=features_cols,
#    outputCol="features"
#)

#df = assembler.transform(df)

# =========================
# PREDICTION (SAFE)
# =========================
df = model.transform(df)

# =========================
# ALERT
# =========================
df = df \
    .withColumn(
        "risk_level",
        F.when(F.col("prediction") == 1.0, "HIGH").otherwise("NORMAL")
    ) \
    .withColumn(
        "alert",
        F.when(
            (F.col("prediction") == 1.0) &
            (
                (F.col("is_low_sbp") == 1.0) |
                (F.col("is_high_hr") == 1.0) |
                (F.col("is_low_spo2") == 1.0)
            ),
            "🚨 CRITICAL"
        ).otherwise("OK")
    )

# =========================
# WRITE
# =========================
query = df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT) \
    .trigger(availableNow=True) \
    .toTable(GOLD_TABLE)

query.awaitTermination()